In [157]:
# ==========================================
# 1. Imports and Configuration
# ==========================================

import os
from pathlib import Path

# Core data manipulation and numerical operations
import pandas as pd
import numpy as np

# Modeling utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest

# Scientific computing
from scipy.stats import uniform

# Bayesian Linear Regression model
# ARDRegression is excellent for handling datasets where many features might be irrelevant
from sklearn.linear_model import ARDRegression

# Bayesian Hyperparameter Optimization
#from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

# Project Configuration
import warnings
warnings.filterwarnings("ignore")

# Set seed for reproducibility if needed later
RANDOM_STATE = 42

In [158]:
# ==========================================
# 2. Data Loading
# ==========================================

# Define the data path.
# the data file is in the 'data/' folder relative to this notebook.


# Define the path to the CPI dataset excel file relative to the notebook's location.
# Since Path(__file__) doesn't work for Jupyter notebooks. We will use the current working directory as a base.
DATA_PATH = Path.cwd().parent.parent / "data" /"raw" / "CPI_latest.xlsx"

def load_cpi_data(file_path: str) -> pd.DataFrame:
    """
    Loads the CPI dataset from the specified Excel file path.

    Args:
        file_path (str): The local or absolute path to the .xlsx file.

    Returns:
        pd.DataFrame: The loaded CPI dataset.
        
    Raises:
        FileNotFoundError: If the dataset is not found at the specified path.
    """
    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}. Please check your path.")
    
    try:
        df = pd.read_excel(file_path)
        print(f"Data loaded successfully. Shape: {df.shape}")
        return df
    except Exception as e:
        print(f"An error occurred while loading the data: {e}")
        return pd.DataFrame()



In [159]:
# Execute data loading
download_cpi = load_cpi_data(DATA_PATH)

# Display the first few rows to confirm structure
download_cpi.head(5)

Data loaded successfully. Shape: (789, 228)


,H01,H02,H03,H04,H05,H06,H13,H17,H18,H24,...,MO042025,MO052025,MO062025,MO072025,MO082025,MO092025,MO102025,MO112025,MO122025,MO012026
0,P0141,Consumer Price Index,CPI60001,Geographic indices,NaN,NaN,Total country,Index,Dec 2024 = 100,2008 01,...,101.9,102.1,102.4,103.3,103.2,103.4,103.5,103.4,103.6,103.7
1,P0141,Consumer Price Index,CPI60051,Geographic indices,NaN,NaN,Rural areas,Index,Dec 2024 = 100,2008 01,...,101.8,102.1,102.4,103.2,103.2,103.2,103.1,103.2,103.3,103.4
2,P0141,Consumer Price Index,CPI60065,Analytical series - All urban areas,CPI for pensioners,NaN,All urban areas,Index,Dec 2024 = 100,2008 01,...,102.1,102.3,102.6,103.6,103.6,103.8,103.9,103.9,104.1,104.2
3,P0141,Consumer Price Index,CPI61001,Geographic indices,NaN,NaN,Western Cape,Index,Dec 2024 = 100,2008 01,...,102.2,102.4,102.5,103.2,103.1,103.5,103.7,103.6,103.8,103.9
4,P0141,Consumer Price Index,CPI62001,Geographic indices,NaN,NaN,Eastern Cape,Index,Dec 2024 = 100,2008 01,...,101.6,102.0,102.3,102.9,102.9,102.9,102.9,102.9,103.1,103.1


In [160]:
# ==========================================
# 3. Data Filtering and Cleaning
# ==========================================

# List of target categories required for the nowcasting competition
TARGET_CATEGORIES = [
    'CPI Headline',
    'Food and non alcoholic beverages',
    'Alcoholic beverages and tobacco',
    'Clothing and footwear',
    'Housing and utilities',
    'Health',
    'Transport',
    'Information and communication',
    'Recreation, sport and culture',
    'Education',
    'Restaurants and accommodation services'
]

def filter_cpi_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters the raw CPI dataframe for the specific categories and urban areas.
    Renames columns for better interpretability.

    Args:
        df (pd.DataFrame): The raw dataframe from Stats SA.

    Returns:
        pd.DataFrame: A filtered and renamed dataframe.
    """
    # Filtering based on specific Stats SA codes:
    # H04: Category name
    # H13: Geographical area (All urban areas is the national benchmark)
    df_filtered = df[
        (df['H04'].isin(TARGET_CATEGORIES)) & 
        (df['H13'] == 'All urban areas')
    ].copy()

    # Mapping internal codes to human-readable names for cleaner code downstream
    rename_map = {
        'H04': 'Category',
    }
    
    # Selecting and renaming specific columns (Adjust these based on your specific Excel structure)
    df_filtered = df_filtered.rename(columns=rename_map)

    return df_filtered

# Apply the filter
cpi_filtered = filter_cpi_data(download_cpi)

# Validate results
print(f"Categories selected: {len(cpi_filtered['Category'].unique())} of {len(TARGET_CATEGORIES)}")
print(cpi_filtered['Category'].unique())

Categories selected: 11 of 11
['CPI Headline' 'Food and non alcoholic beverages'
 'Alcoholic beverages and tobacco' 'Clothing and footwear'
 'Housing and utilities' 'Health' 'Transport'
 'Information and communication' 'Recreation, sport and culture'
 'Education' 'Restaurants and accommodation services']


In [161]:
# ==========================================
# 4. Refining Category Granularity
# ==========================================

def refine_education_category(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filters out specific sub-categories within the 'Education' group to 
    ensure we are only using the primary 'Education' index.
    
    This prevents double-counting or using inconsistent sub-indices.

    Args:
        df (pd.DataFrame): The filtered CPI dataframe.

    Returns:
        pd.DataFrame: Dataframe with unwanted education sub-categories removed.
    """
    # first ensure there are more than one unique instances of 'Education' in the Category column
    if df[df['Category'] == 'Education'].shape[0] <= 1:
        print("No multiple sub-categories found for 'Education'.")

        #Dropping extra features that aren't needed
        df_refined = df.drop(['H01','H02','H03','H05','H06','H13',
                                    'H17','H18','H24','H25'],axis=1)
        return df_refined
    
    # List of sub-categories to remove as identified in the H05 column
    # These represent granular components that might deviate from the main index
    unwanted_edu_subcats = [
        'University boarding fees',
        'Tertiary education',
        'Primary and secondary education',
        'Education including boarding fees'
    ]

    
    df_refined = df[~(df['H05'].isin(unwanted_edu_subcats))].copy()

    #Dropping extra features that aren't needed
    df_refined = df_refined.drop(['H01','H02','H03','H05','H06','H13',
                                  'H17','H18','H24','H25'],axis=1)
    
    initial_count = len(df)
    final_count = len(df_refined)
    
    print(f"Refinement complete: Removed {initial_count - final_count} sub-category rows.")
    
    return df_refined


edu_filtered = refine_education_category(cpi_filtered)

No multiple sub-categories found for 'Education'.


In [162]:
edu_filtered.head(3)

,Category,MO012008,MO022008,MO032008,MO042008,MO052008,MO062008,MO072008,MO082008,MO092008,...,MO042025,MO052025,MO062025,MO072025,MO082025,MO092025,MO102025,MO112025,MO122025,MO012026
12,CPI Headline,41.9,42.2,42.8,43.0,43.3,43.9,44.5,44.8,45.0,...,101.9,102.1,102.4,103.3,103.2,103.4,103.5,103.4,103.6,103.8
45,Food and non alcoholic beverages,34.3,34.3,34.8,35.3,35.9,36.5,37.0,37.5,38.1,...,102.2,103.3,104.0,104.6,104.5,104.3,104.1,104.2,104.4,104.8
59,Alcoholic beverages and tobacco,35.3,35.7,37.3,37.5,37.6,37.6,37.7,38.3,38.7,...,103.3,103.5,103.8,104.0,104.0,104.1,104.7,104.8,104.6,104.8


In [163]:
# ==========================================
# 5. Time-Series Structural Alignment
# ==========================================

def format_cpi_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates a standardized date range and assigns it as column headers.
    
    Args:
        df (pd.DataFrame): The filtered CPI dataframe.

    Returns:
        pd.DataFrame: Dataframe with formatted date columns.
    """
    # extract start mmyyyy and end mmyyyy from dataframe columns such that df.columns[1]
    # is the start date in the format 'MOmmyyyy' and df.columns[-1] is the end date in the same format
    start_date_str = df.columns[1]
    end_date_str = df.columns[-1]
    start_date = pd.to_datetime(start_date_str, format='MO%m%Y')
    end_date = pd.to_datetime(end_date_str, format='MO%m%Y')
    
    # Generate a sequence of dates from start to end with monthly frequency
    date_index = pd.date_range(start=start_date, end=end_date, freq='MS')

    
    formatted_dates = [f"{d.month}-{d.year}" for d in date_index]

    # Create the new column list
    new_columns = ['Category'] + formatted_dates

    print(f"Generated {len(formatted_dates)} date columns from {start_date_str} to {end_date_str}.")
    print(f"Expected columns: {new_columns[:5]} ... {new_columns[-5:]}")  # Show a sample of the new columns
    
    # Validation: Ensure column count matches data shape
    if len(new_columns) != df.shape[1]:
        print(f"Warning: Column mismatch. Data has {df.shape[1]} columns, but we generated {len(new_columns)}.")
    
    df.columns = new_columns
    return df

In [164]:
# Execute column renaming
edu_filtered = format_cpi_columns(edu_filtered)

# Check the last few columns to ensure it ends at 7-2023
print(f"Dataset structure: {edu_filtered.shape}")
print(f"Last column: {edu_filtered.columns[-1]}")

Generated 217 date columns from MO012008 to MO012026.
Expected columns: ['Category', '1-2008', '2-2008', '3-2008', '4-2008'] ... ['9-2025', '10-2025', '11-2025', '12-2025', '1-2026']
Dataset structure: (11, 218)
Last column: 1-2026


In [165]:
edu_filtered

,Category,1-2008,2-2008,3-2008,4-2008,5-2008,6-2008,7-2008,8-2008,9-2008,...,4-2025,5-2025,6-2025,7-2025,8-2025,9-2025,10-2025,11-2025,12-2025,1-2026
12,CPI Headline,41.9,42.2,42.8,43.0,43.3,43.9,44.5,44.8,45.0,...,101.9,102.1,102.4,103.3,103.2,103.4,103.5,103.4,103.6,103.8
45,Food and non alcoholic beverages,34.3,34.3,34.8,35.3,35.9,36.5,37.0,37.5,38.1,...,102.2,103.3,104.0,104.6,104.5,104.3,104.1,104.2,104.4,104.8
59,Alcoholic beverages and tobacco,35.3,35.7,37.3,37.5,37.6,37.6,37.7,38.3,38.7,...,103.3,103.5,103.8,104.0,104.0,104.1,104.7,104.8,104.6,104.8
65,Clothing and footwear,61.5,61.6,61.8,61.8,62.0,62.3,62.4,62.8,63.0,...,100.5,100.6,100.7,100.8,100.9,101.0,101.1,101.1,101.2,101.3
68,Housing and utilities,40.2,40.2,40.8,40.9,40.9,41.6,42.6,42.9,43.3,...,100.6,100.7,101.2,103.7,103.7,104.3,104.3,104.3,104.9,104.9
79,Health,38.8,40.7,40.7,40.7,41.1,41.2,41.3,41.4,41.4,...,104.0,104.2,104.5,104.4,104.6,104.9,105.0,105.0,105.0,105.2
82,Transport,45.8,45.9,46.8,47.4,47.9,49.5,50.5,50.0,49.7,...,100.3,100.1,100.0,100.7,100.5,100.6,100.7,100.3,101.0,99.9
89,Information and communication,106.6,106.5,106.4,106.0,106.1,106.1,106.0,107.3,107.1,...,101.5,101.5,101.5,101.4,101.2,101.0,100.7,100.3,100.2,100.2
92,"Recreation, sport and culture",63.6,63.6,63.6,64.2,65.2,65.2,65.4,66.7,66.9,...,101.6,101.6,101.7,102.3,102.7,102.8,103.3,102.6,102.5,102.8
97,Education,30.5,30.5,32.6,32.6,32.6,32.6,32.6,32.6,32.6,...,104.5,104.5,104.5,104.5,104.5,104.5,104.5,104.5,104.5,104.5


In [166]:
# ==========================================
# 6. Reshaping to Long Format
# ==========================================

def reshape_cpi_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivots the dataframe from wide (months as columns) to long (months as rows).
    Converts dates to datetime objects and sorts the data chronologically.

    Args:
        df (pd.DataFrame): The wide-format CPI dataframe.

    Returns:
        pd.DataFrame: A cleaned, long-format dataframe sorted by Category and Date.
    """
    # 1. Melt the data: 'Category' stays, all date columns become rows in 'Date'
    df_long = pd.melt(df, id_vars='Category', var_name='Date', value_name='Value')

    # 2. Convert 'Date' column to datetime objects
    # Our format was 'Month-Year' (e.g., '1-2008')
    df_long['Date'] = pd.to_datetime(df_long['Date'], format='%m-%Y')

    # 3. Ensure 'Value' is numeric (important for modeling)
    df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

    # 4. Sort the data
    # Sorting by Category then Date ensures a clean sequence for feature engineering
    df_long = df_long.sort_values(by=['Category', 'Date']).reset_index(drop=True)

    return df_long

# Apply the transformation
cpi_long = reshape_cpi_data(edu_filtered)

# Preview the results
print(f"Long format shape: {cpi_long.shape}")
print(cpi_long.head())

Long format shape: (2387, 3)
                          Category       Date  Value
0  Alcoholic beverages and tobacco 2008-01-01   35.3
1  Alcoholic beverages and tobacco 2008-02-01   35.7
2  Alcoholic beverages and tobacco 2008-03-01   37.3
3  Alcoholic beverages and tobacco 2008-04-01   37.5
4  Alcoholic beverages and tobacco 2008-05-01   37.6


In [167]:
# ==========================================
# 7. Feature Engineering (Lagged Observations)
# ==========================================

def create_lagged_features(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    """
    Creates auto-regressive features by shifting the CPI values.
    
    This function treats each CPI category as an independent time series 
    to prevent data leakage between different categories.

    Args:
        df (pd.DataFrame): Long-format dataframe (Expected order: Descending by Date).
        steps (int): The number of previous months to use as features.

    Returns:
        pd.DataFrame: Dataframe with new columns 'Value_1', 'Value_2', etc., 
                      representing past observations.
    """
    data = df.sort_values(by=['Category', 'Date'], ascending=[True, True]).copy()  # Ensure correct order for shifting
    
    # We group by 'Category' to ensure shifts happen within the same CPI class.
    # We use .shift(i) because the data is sorted with the oldest dates at the top.
    for i in range(1, steps + 1):
        data[f'Value_{i}'] = data.groupby('Category')['Value'].shift(i)

    # Drop rows with NaN values created by the shift (the oldest 'steps' months)
    data = data.dropna(axis=0)
    
    # Reset index for a clean slate before modeling
    data = data.reset_index(drop=True)
    
    print(f"Feature engineering complete. Created {steps} lag features.")
    print(f"New shape: {data.shape}")
    
    return data

# Generating 15 months of lags
LAG_STEPS = 15
cpi_with_lags = create_lagged_features(cpi_long, steps=LAG_STEPS)

# Preview the head to see Value_1, Value_2... alongside the target 'Value'
cpi_with_lags.head()

Feature engineering complete. Created 15 lag features.
New shape: (2222, 18)


,Category,Date,Value,Value_1,Value_2,Value_3,Value_4,Value_5,Value_6,Value_7,Value_8,Value_9,Value_10,Value_11,Value_12,Value_13,Value_14,Value_15
0,Alcoholic beverages and tobacco,2009-04-01,41.6,41.2,39.3,38.9,38.9,38.9,38.8,38.7,38.3,37.7,37.6,37.6,37.5,37.3,35.7,35.3
1,Alcoholic beverages and tobacco,2009-05-01,41.6,41.6,41.2,39.3,38.9,38.9,38.9,38.8,38.7,38.3,37.7,37.6,37.6,37.5,37.3,35.7
2,Alcoholic beverages and tobacco,2009-06-01,41.6,41.6,41.6,41.2,39.3,38.9,38.9,38.9,38.8,38.7,38.3,37.7,37.6,37.6,37.5,37.3
3,Alcoholic beverages and tobacco,2009-07-01,42.1,41.6,41.6,41.6,41.2,39.3,38.9,38.9,38.9,38.8,38.7,38.3,37.7,37.6,37.6,37.5
4,Alcoholic beverages and tobacco,2009-08-01,43.1,42.1,41.6,41.6,41.6,41.2,39.3,38.9,38.9,38.9,38.8,38.7,38.3,37.7,37.6,37.6


In [168]:
# ==========================================
# 8. Feature Engineering (Time Series Summary Statistics)
# ==========================================

def vectorized_trend(lag_data):
    """Calculates the linear trend slope across the lags."""
    n = lag_data.shape[1]
    if n < 2: return np.zeros(len(lag_data))
    x = np.arange(n)
    x_mean = x.mean()
    # (sum(xi - xmean)(yi - ymean)) / sum(xi - xmean)^2
    numerator = ((lag_data.subtract(lag_data.mean(axis=1), axis=0))
                 .multiply(x - x_mean, axis=1)).sum(axis=1)
    denominator = ((x - x_mean)**2).sum()
    return numerator / denominator

def ts_stats_features(df: pd.DataFrame, steps: int) -> pd.DataFrame:
    """
    Creates auto-regressive features by shifting the CPI values.
    
    This function treats each CPI category as an independent time series 
    to prevent data leakage between different categories.

    Args:
        df (pd.DataFrame): Long-format dataframe (Expected order: Descending by Date).
        steps (int): The number of previous months to use as features.

    Returns:
        pd.DataFrame: Dataframe with new columns 'Value_1', 'Value_2', etc., 
                      representing past observations.
    """
    lagged_data = df[[f'Value_{i}' for i in range(1, steps + 1)]]
    
    df_dict = {}
    df_dict[f'past_{steps}_mean'] = lagged_data.mean(axis=1)
    df_dict[f'past_{steps}_std'] = lagged_data.std(axis=1)
    df_dict[f'past_{steps}_min'] = lagged_data.min(axis=1)
    df_dict[f'past_{steps}_max'] = lagged_data.max(axis=1)
    df_dict[f'past_{steps}_median'] = lagged_data.median(axis=1)
    df_dict[f'past_{steps}_skew'] = lagged_data.skew(axis=1)
    df_dict[f'past_{steps}_kurt'] = lagged_data.kurtosis(axis=1)
    df_dict[f'past_{steps}_trend'] = vectorized_trend(lagged_data)
    df_dict[f'past_{int(steps/2)}_trend'] = vectorized_trend(lagged_data.iloc[:, :int(steps/2)])
    
    df_stats = pd.DataFrame(df_dict)

    # Concatenate the original dataframe with the new statistics dataframe
    data = pd.concat([df, df_stats], axis=1)
    
    print(f"Time series summary statistics generated.")
    print(f"New shape: {data.shape}")
    
    return data

cpi_with_lag_stats = ts_stats_features(cpi_with_lags, steps=LAG_STEPS)

# Preview the head
cpi_with_lag_stats.head()

Time series summary statistics generated.
New shape: (2222, 27)


,Category,Date,Value,Value_1,Value_2,Value_3,Value_4,Value_5,Value_6,Value_7,...,Value_15,past_15_mean,past_15_std,past_15_min,past_15_max,past_15_median,past_15_skew,past_15_kurt,past_15_trend,past_7_trend
0,Alcoholic beverages and tobacco,2009-04-01,41.6,41.2,39.3,38.9,38.9,38.9,38.8,38.7,...,35.3,38.113333,1.442154,35.3,41.2,38.3,-0.119824,1.097273,-0.299286,-0.303571
1,Alcoholic beverages and tobacco,2009-05-01,41.6,41.6,41.2,39.3,38.9,38.9,38.9,38.8,...,35.7,38.533333,1.481151,35.7,41.6,38.7,0.519538,1.010127,-0.306071,-0.478571
2,Alcoholic beverages and tobacco,2009-06-01,41.6,41.6,41.6,41.2,39.3,38.9,38.9,38.9,...,37.3,38.926667,1.458212,37.3,41.6,38.8,0.957636,-0.156691,-0.301786,-0.564286
3,Alcoholic beverages and tobacco,2009-07-01,42.1,41.6,41.6,41.6,41.2,39.3,38.9,38.9,...,37.5,39.213333,1.536167,37.5,41.6,38.9,0.697354,-0.983802,-0.322143,-0.564286
4,Alcoholic beverages and tobacco,2009-08-01,43.1,42.1,41.6,41.6,41.6,41.2,39.3,38.9,...,37.6,39.520000,1.626214,37.6,42.1,38.9,0.484111,-1.412974,-0.345357,-0.521429


In [169]:
def data_split(df,train_size,date_col='Date'):
    """
    Splits the dataframe into features and predictors for 
    training and validation sets based on a specified date column.

    Args:
        df (pd.DataFrame): The input dataframe to split.
        train_size (float): The proportion of the data to include in the training set (between 0 and 1).
        date_col (str): The name of the date column to use for splitting.

    Returns:
        pd.DataFrame,pd.Series,pd.DataFrame,pd.Series: The training and validation features and targets.
    """
    # Ensure the date column is in datetime format
    df[date_col] = pd.to_datetime(df[date_col])

    split_date = df[date_col].quantile(train_size)

    # Split the dataframe into training and validation sets
    train_df = df[df[date_col] < split_date].reset_index(drop=True)
    valid_df = df[df[date_col] >= split_date].reset_index(drop=True)

    print(f"Data split at date: {split_date}. Training set: {train_df.shape}, Validation set: {valid_df.shape}")

    # separate features and target
    X_train = train_df.drop(['Value', date_col,'Category'], axis=1)
    y_train = train_df['Value']
    X_valid = valid_df.drop(['Value', date_col,'Category'], axis=1)
    y_valid = valid_df['Value']
    
    return X_train, y_train, X_valid, y_valid

# Perform the split
X_train, y_train, X_valid, y_valid = data_split(cpi_with_lag_stats, train_size=0.8)

Data split at date: 2022-09-01 00:00:00. Training set: (1771, 27), Validation set: (451, 27)


In [170]:
from catboost import Pool, CatBoostRegressor
from sklearn.metrics import root_mean_squared_error
CAT_COLS = []

In [171]:
# ==========================================
# 8. Feature Selection (Catboost's built-in feature importance)
# ==========================================

def reg_objective_unk(Xtrain,ytrain,Xtest,ytest,no_remove,
                       rem=5,steps=6,iter=100,cat_cols=CAT_COLS):
    
    
    obj = 'RMSE'
    
    cols_to_eliminate = {}
    step_lls = {}
    for step in range(steps):
        col_oof_ll = []
        oof_unimp = []
        if step==0:
            npyc_df = Xtrain
        else:
            drop_cols = cols_to_eliminate[f'step_{step-1}']
            npyc_df = Xtrain.drop(drop_cols,axis=1)

        step_level_imps = {k:[] for k in npyc_df.columns}
        for Xs, ys in [
            [(Xtrain[npyc_df.columns],Xtest[npyc_df.columns]),
             (ytrain.copy(),ytest.copy())]
        ]:

            X_train, X_test = Xs
            y_train, y_test = ys

            
            #display(y_train.value_counts())
            params = {'iterations': iter,'learning_rate': 0.05,'depth': 5,
                      "loss_function":obj,
                      #'objective': obj,
                      'eval_metric': obj,
                      'random_seed': 42,'verbose': int(iter/2),
                      'task_type':'CPU','use_best_model':True}
            
            
            
            prepped_X =X_train 
            prepped_Xtest =X_test
            

            cat_features = [i for i, col in enumerate(X_train.columns) if col in cat_cols]
            
            train_pool = Pool(prepped_X, y_train,cat_features=cat_features)
            test_pool = Pool(prepped_Xtest, y_test, cat_features=cat_features)
            
            model = CatBoostRegressor(**params)

            model.fit(train_pool,eval_set=test_pool)

            feature_importances = model.get_feature_importance()
            

            for i,col in enumerate(X_train.columns):
                step_level_imps[col].append(feature_importances[i])

            
            pred = model.predict(prepped_Xtest)
            
            ind_cv = root_mean_squared_error(y_test, pred)
            
            col_oof_ll.append(ind_cv)
            
        print(f"rmse step {step}:  {np.mean(col_oof_ll)}")
        step_lls[step] = np.mean(col_oof_ll)

        step_imps_avg = {k:np.mean(v) for k,v in step_level_imps.items()}
        importance_df = pd.DataFrame({
            'Feature': step_imps_avg.keys(),
            'Importance': step_imps_avg.values()
            }).sort_values(by='Importance', ascending=False)

        #print('min',importance_df['Importance'].min())
        #print('max',importance_df['Importance'].max())
        print('top 5: ')
        display(importance_df.head())

        #print('bottom 5: ')
        #display(importance_df.tail())
        importance_df = importance_df[~(importance_df['Feature'].isin(no_remove))]
        oof_unimp = importance_df[importance_df['Importance']<=0]['Feature'].to_list()
        min_rem_size = rem
        if len(oof_unimp)<min_rem_size:
            
            oof_unimp = importance_df.tail(min_rem_size)['Feature'].to_list()
            #oof_unimp = [col for col in oof_unimp if col not in no_remove]
        print('to remove: ',oof_unimp)
        if step==0:
            cols_to_eliminate['step_0'] = oof_unimp
        elif step<steps-1:
            cols_to_eliminate[f'step_{step}'] = cols_to_eliminate[f'step_{step-1}'] + oof_unimp
             
    min_u = min(step_lls.values())
    chosen_step = [x for x in step_lls.keys() if step_lls[x]==min_u]
    chosen_step = chosen_step[-1]
    if chosen_step>0:
        chosen_features = [col for col in Xtrain.columns if col not in cols_to_eliminate[f'step_{chosen_step-1}']]
    else:
        chosen_features = Xtrain.columns.tolist()
    print(f'chosen step is {chosen_step} with value: {min_u}')
    return chosen_features, step_lls, cols_to_eliminate


#best mae 1.315314744746251
sel_feats, lls,cols = reg_objective_unk(X_train,y_train,
                            X_valid,y_valid,[],
                            1,X_train.shape[1]-5,100)

0:	learn: 16.4179322	test: 26.7837266	best: 26.7837266 (0)	total: 16.6ms	remaining: 1.65s
50:	learn: 1.9670638	test: 5.2400187	best: 5.2400187 (50)	total: 313ms	remaining: 301ms
99:	learn: 0.8676372	test: 3.3270993	best: 3.3270993 (99)	total: 534ms	remaining: 0us

bestTest = 3.327099275
bestIteration = 99

rmse step 0:  3.3270991747676417
top 5: 


,Feature,Importance
3,Value_4,10.014718
19,past_15_median,9.270447
15,past_15_mean,8.881525
4,Value_5,8.343252
1,Value_2,7.257737


to remove:  ['past_15_skew']
0:	learn: 16.4104892	test: 26.8260467	best: 26.8260467 (0)	total: 5.25ms	remaining: 519ms
50:	learn: 1.9931813	test: 4.9488920	best: 4.9488920 (50)	total: 254ms	remaining: 244ms
99:	learn: 0.9022166	test: 2.8284476	best: 2.8284476 (99)	total: 535ms	remaining: 0us

bestTest = 2.82844761
bestIteration = 99

rmse step 1:  2.828447555946062
top 5: 


,Feature,Importance
0,Value_1,20.514149
15,past_15_mean,9.206505
3,Value_4,7.614920
1,Value_2,6.820363
18,past_15_max,6.304284


to remove:  ['past_15_kurt']
0:	learn: 16.4029367	test: 26.3856109	best: 26.3856109 (0)	total: 10.8ms	remaining: 1.07s
50:	learn: 1.9685602	test: 4.0646232	best: 4.0646232 (50)	total: 380ms	remaining: 365ms
99:	learn: 0.8873762	test: 2.1108141	best: 2.1108141 (99)	total: 596ms	remaining: 0us

bestTest = 2.110814109
bestIteration = 99

rmse step 2:  2.110814015986946
top 5: 


...

to remove:  ['Value_5']
0:	learn: 16.4172938	test: 26.5139470	best: 26.5139470 (0)	total: 6.43ms	remaining: 637ms
50:	learn: 1.8984266	test: 3.9913778	best: 3.9913778 (50)	total: 235ms	remaining: 226ms
99:	learn: 0.9493345	test: 2.5073897	best: 2.5073897 (99)	total: 405ms	remaining: 0us

bestTest = 2.507389711
bestIteration = 99

rmse step 17:  2.5073896374076963
top 5: 


,Feature,Importance
5,past_15_max,21.482149
3,Value_12,19.909685
2,Value_11,15.160191
6,past_15_median,13.274774
1,Value_3,12.857095


to remove:  ['past_15_mean']
0:	learn: 16.3952142	test: 26.8825845	best: 26.8825845 (0)	total: 4.63ms	remaining: 459ms
50:	learn: 1.9363229	test: 3.7860659	best: 3.7860659 (50)	total: 161ms	remaining: 155ms
99:	learn: 0.9642099	test: 2.2266820	best: 2.2266820 (99)	total: 281ms	remaining: 0us

bestTest = 2.226682028
bestIteration = 99

rmse step 18:  2.226681989349467
top 5: 


,Feature,Importance
1,Value_3,23.969279
2,Value_11,16.673295
0,Value_1,16.259360
5,past_15_median,15.309740
4,past_15_max,14.385125


to remove:  ['Value_12']
chosen step is 14 with value: 2.0855070163361287


In [172]:
print(sel_feats)

['Value_1', 'Value_3', 'Value_5', 'Value_7', 'Value_10', 'Value_11', 'Value_12', 'past_15_mean', 'past_15_max', 'past_15_median']


In [173]:
import optuna

In [174]:
def reg_objective(trial, X_train, y_train, X_test, y_test,cat_cols=CAT_COLS):
    # --- 1. Basic Tree Params ---
    iterations = trial.suggest_int("iterations", 100, 1000)
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.3, log=True)
    depth = trial.suggest_int("depth", 4, 10)
    l2_leaf_reg = trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True)
    
    # --- 2. Randomness & Noise (New) ---
    random_strength = trial.suggest_float("random_strength", 1e-9, 10.0, log=True)
    
    # --- 3. Tree Growth Strategy (New) ---
    grow_policy = trial.suggest_categorical("grow_policy", ["SymmetricTree", "Depthwise", "Lossguide"])
    
    # Min data in leaf interacts differently depending on grow_policy, 
    # but generally safe to tune.
    min_data_in_leaf = trial.suggest_int("min_data_in_leaf", 1, 100) 

    # If Lossguide, we need to limit the number of leaves specifically
    if grow_policy == "Lossguide":
        max_leaves = trial.suggest_int("max_leaves", 16, 64)
    else:
        max_leaves = None # Ignored for SymmetricTree/Depthwise


    # --- 5. Bootstrapping (Fixed Logic) ---
    bootstrap_type = trial.suggest_categorical("bootstrap_type", ['Bayesian', 'Bernoulli', 'MVS'])
    
    subsample = None
    bagging_temperature = None
    
    if bootstrap_type in ['Bernoulli', 'MVS']:
        subsample = trial.suggest_float("subsample", 0.5, 1.0)
    elif bootstrap_type == 'Bayesian':
        bagging_temperature = trial.suggest_float("bagging_temperature", 0.0, 10.0)

    # --- 6. Feature Sampling ---
    colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.4, 1.0)

    
    cat_features = [i for i, col in enumerate(X_train.columns) if col in cat_cols]
    train_pool = Pool(X_train, y_train, cat_features=cat_features)
    test_pool = Pool(X_test, y_test, cat_features=cat_features)

    
    model = CatBoostRegressor(
        iterations=iterations,  
        learning_rate=learning_rate,
        depth=depth,
        l2_leaf_reg=l2_leaf_reg,
        
        # New params
        random_strength=random_strength,
        grow_policy=grow_policy,
        max_leaves=max_leaves,
        
        # Bootstrap params
        bootstrap_type=bootstrap_type,
        subsample=subsample,
        bagging_temperature=bagging_temperature,
        
        colsample_bylevel=colsample_bylevel,
        min_data_in_leaf=min_data_in_leaf,
        
        objective='RMSE',
        task_type='CPU',
        random_state=42,
        silent=True,
        use_best_model=True,
        eval_metric='RMSE'
    )

    model.fit(train_pool, eval_set=test_pool)
    
    y_pred = model.predict(X_test)
    score = root_mean_squared_error(y_test, y_pred)
    

    return score

In [175]:
def tune_hyperparameters(X_train, y_train, X_valid, y_valid, sel_feats,trials=100, timeout=43200):
    rstudy = optuna.create_study(direction='minimize')
    rstudy.optimize(lambda trial: reg_objective(trial,X_train[sel_feats],y_train,
                                                X_valid[sel_feats],y_valid),
                gc_after_trial=True,n_trials=trials, timeout=timeout)
    
    return rstudy.best_params

hyp = tune_hyperparameters(X_train, y_train, X_valid, y_valid, sel_feats,trials=100)

[I 2026-03-10 19:05:51,188] A new study created in memory with name: no-name-26f7d99a-2f2f-4104-9ce3-2ebdcab2994c
[I 2026-03-10 19:06:30,304] Trial 0 finished with value: 2.0975524866448354 and parameters: {'iterations': 897, 'learning_rate': 0.002820498992533182, 'depth': 10, 'l2_leaf_reg': 0.02315044155539081, 'random_strength': 4.657753479854167e-08, 'grow_policy': 'SymmetricTree', 'min_data_in_leaf': 12, 'bootstrap_type': 'MVS', 'subsample': 0.8051121600932098, 'colsample_bylevel': 0.7703749235326827}. Best is trial 0 with value: 2.0975524866448354.
[I 2026-03-10 19:06:34,078] Trial 1 finished with value: 1.1868497260321857 and parameters: {'iterations': 322, 'learning_rate': 0.09370959395352477, 'depth': 8, 'l2_leaf_reg': 3.4732862887448537, 'random_strength': 8.098764479552818e-09, 'grow_policy': 'Depthwise', 'min_data_in_leaf': 9, 'bootstrap_type': 'MVS', 'subsample': 0.7469973563281365, 'colsample_bylevel': 0.6217221874009061}. Best is trial 1 with value: 1.1868497260321857.
[I

In [176]:
def fit_model(hyperparams,X_train,y_train,X_test,y_test,
              cat_cols=CAT_COLS,sel_feats=sel_feats):
    '''fit model on all of the available data'''
    #combine predictors and features of train and test into one set of predictors and features
    X = pd.concat([X_train,X_test]).reset_index(drop=True)
    y = pd.concat([y_train,y_test]).reset_index(drop=True)

    obj = 'RMSE'
    params = {'objective': obj,
            'random_seed': 42,'silent': True,
            'task_type':'CPU',**hyperparams}



    prepped_X =X[sel_feats] 

    cat_features = [i for i, col in enumerate(sel_feats) if col in cat_cols]

    train_pool = Pool(prepped_X, y,cat_features=cat_features)

    model = CatBoostRegressor(**params)
    model.fit(train_pool)

    return model


cpi_model = fit_model(hyp,X_train,y_train,X_valid,
                      y_valid,)

# Preparing Submission

In [177]:
cpi_with_lag_stats#['Date'].max()

,Category,Date,Value,Value_1,Value_2,Value_3,Value_4,Value_5,Value_6,Value_7,...,Value_15,past_15_mean,past_15_std,past_15_min,past_15_max,past_15_median,past_15_skew,past_15_kurt,past_15_trend,past_7_trend
0,Alcoholic beverages and tobacco,2009-04-01,41.6,41.2,39.3,38.9,38.9,38.9,38.8,38.7,...,35.3,38.113333,1.442154,35.3,41.2,38.3,-0.119824,1.097273,-0.299286,-0.303571
1,Alcoholic beverages and tobacco,2009-05-01,41.6,41.6,41.2,39.3,38.9,38.9,38.9,38.8,...,35.7,38.533333,1.481151,35.7,41.6,38.7,0.519538,1.010127,-0.306071,-0.478571
2,Alcoholic beverages and tobacco,2009-06-01,41.6,41.6,41.6,41.2,39.3,38.9,38.9,38.9,...,37.3,38.926667,1.458212,37.3,41.6,38.8,0.957636,-0.156691,-0.301786,-0.564286
3,Alcoholic beverages and tobacco,2009-07-01,42.1,41.6,41.6,41.6,41.2,39.3,38.9,38.9,...,37.5,39.213333,1.536167,37.5,41.6,38.9,0.697354,-0.983802,-0.322143,-0.564286
4,Alcoholic beverages and tobacco,2009-08-01,43.1,42.1,41.6,41.6,41.6,41.2,39.3,38.9,...,37.6,39.520000,1.626214,37.6,42.1,38.9,0.484111,-1.412974,-0.345357,-0.521429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2217,Transport,2025-09-01,100.6,100.5,100.7,100.0,100.1,100.3,101.3,101.2,...,103.4,100.760000,1.119183,99.2,103.4,100.5,1.035280,0.885000,0.123214,0.128571
2218,Transport,2025-10-01,100.7,100.6,100.5,100.7,100.0,100.1,100.3,101.3,...,102.4,100.573333,0.848079,99.2,102.4,100.5,0.667727,0.360757,0.051786,0.039286
2219,Transport,2025-11-01,100.3,100.7,100.6,100.5,100.7,100.0,100.1,100.3,...,101.9,100.460000,0.684314,99.2,101.9,100.5,0.282826,0.389603,-0.003571,-0.096429
2220,Transport,2025-12-01,101.0,100.3,100.7,100.6,100.5,100.7,100.0,100.1,...,100.7,100.353333,0.556605,99.2,101.3,100.3,-0.231444,0.232622,-0.040714,-0.067857


In [178]:
# ==========================================
# 11. Inference Preparation (August 2023)
# ==========================================

def prepare_inference_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Extracts the most recent feature row to be used for nowcasting the next month.
    
    Args:
        df (pd.DataFrame): The dataframe containing lagged features.

    Returns:
        pd.DataFrame: A dataframe containing the features for August 2023 forecast.
    """
    latest_date = df['Date'].max()

    # 1. Filter for the most recent month 
    latest_data = df[df['Date'] == latest_date].copy()
    
    # 2. Re-aligning Lags for the future month:
    
    # Identify all 'Value_i' columns present in the training data
    lag_cols = [col for col in df.columns if col.startswith('Value_')]
    
    # Create the new feature set for inference by shifting the lag columns to represent the next month
    cpi_with_lag = latest_data[['Category','Date', 'Value'] + lag_cols[:-1]].copy()
    
    # Rename columns to match the model's expected input (Value_1, Value_2, etc.)
    new_col_names = ['Category','Date'] + [f'Value_{i}' for i in range(1, len(lag_cols) + 1)]
    cpi_with_lag.columns = new_col_names

    cpi_with_lag['Date'] = cpi_with_lag['Date'] + pd.DateOffset(months=1)

    cpi_with_lag_stats = ts_stats_features(cpi_with_lag, steps=LAG_STEPS)
    
    print(f"Inference features prepared for {len(cpi_with_lag_stats)} categories.")
    return cpi_with_lag_stats

# Note: Using the datetime object we created in step 6
inference_features = prepare_inference_data(cpi_with_lags)

# Display features for a quick sanity check
inference_features.head()

Time series summary statistics generated.
New shape: (11, 26)
Inference features prepared for 11 categories.


,Category,Date,Value_1,Value_2,Value_3,Value_4,Value_5,Value_6,Value_7,Value_8,...,Value_15,past_15_mean,past_15_std,past_15_min,past_15_max,past_15_median,past_15_skew,past_15_kurt,past_15_trend,past_7_trend
201,Alcoholic beverages and tobacco,2026-02-01,104.8,104.6,104.8,104.7,104.1,104.0,104.0,103.8,...,100.2,102.966667,1.851512,100.0,104.8,103.8,-0.749792,-1.205840,-0.384643,-0.153571
403,CPI Headline,2026-02-01,103.8,103.6,103.4,103.5,103.4,103.2,103.3,102.4,...,99.9,102.240000,1.376227,99.9,103.8,102.4,-0.609265,-1.065131,-0.295714,-0.082143
605,Clothing and footwear,2026-02-01,101.3,101.2,101.1,101.1,101.0,100.9,100.8,100.7,...,100.0,100.660000,0.445293,100.0,101.3,100.7,-0.209014,-1.330111,-0.098929,-0.078571
807,Education,2026-02-01,104.5,104.5,104.5,104.5,104.5,104.5,104.5,104.5,...,100.0,103.300000,2.059820,100.0,104.5,104.5,-1.176354,-0.734266,-0.353571,0.000000
1009,Food and non alcoholic beverages,2026-02-01,104.8,104.4,104.2,104.1,104.3,104.5,104.6,104.0,...,99.8,102.820000,1.905706,99.8,104.8,104.0,-0.570313,-1.571205,-0.391786,-0.010714


In [179]:
def get_prediction(df,model):
    '''get prediction for the next month'''
    df['Predicted_Value'] = model.predict(df[sel_feats])
    return df[['Category','Date','Predicted_Value']]

prediction_df = get_prediction(inference_features,cpi_model)

In [180]:
prediction_df

,Category,Date,Predicted_Value
201,Alcoholic beverages and tobacco,2026-02-01,104.776225
403,CPI Headline,2026-02-01,103.887703
605,Clothing and footwear,2026-02-01,101.095110
807,Education,2026-02-01,104.630754
1009,Food and non alcoholic beverages,2026-02-01,104.746613
1211,Health,2026-02-01,105.298013
1413,Housing and utilities,2026-02-01,104.671097
1615,Information and communication,2026-02-01,100.295421
1817,"Recreation, sport and culture",2026-02-01,102.734657
2019,Restaurants and accommodation services,2026-02-01,104.231440
